# Fake News Detection — Notebook 3: BERT Fine-Tuning

Fine-tune `bert-base-uncased` for binary fake-news classification.

**Input:** `data/lemmatized.csv`  
**Output:** `bert-finetuned/` (saved model + tokenizer)

> **Requirements:** `pip install transformers torch`  
> A GPU is strongly recommended (CUDA or Apple MPS). Training on CPU is possible but slow.

---

## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

device = torch.device('cuda' if torch.cuda.is_available() else
                      'mps'  if torch.backends.mps.is_available() else
                      'cpu')
print(f'Using device: {device}')


Using device: mps


## 1. Load Data

In [2]:
df = pd.read_csv('../data/lemmatized.csv')
df = df[['text', 'label']].dropna().reset_index(drop=True)
df['label'] = df['label'].astype(int)

print(f'Samples: {len(df)}')
print(f'Label distribution:\n{df["label"].value_counts()}')


Samples: 71349
Label distribution:
label
1    36323
0    35026
Name: count, dtype: int64


## 2. Tokenizer & Model

In [3]:
MODEL_NAME = 'bert-base-uncased'

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
).to(device)

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model parameters: 109,483,778


## 3. Dataset Class

In [4]:
class FakeNewsDataset(Dataset):
    """Tokenises on-the-fly to avoid storing the full tensor matrix in memory."""

    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts     = list(texts)
        self.labels    = list(labels)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt',
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long),
        }


## 4. Train / Test Split & DataLoaders

In [5]:
BATCH_SIZE = 16
MAX_LEN    = 128

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df['text'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)

train_dataset = FakeNewsDataset(
    train_texts.reset_index(drop=True),
    train_labels.reset_index(drop=True),
    tokenizer, MAX_LEN,
)
test_dataset = FakeNewsDataset(
    test_texts.reset_index(drop=True),
    test_labels.reset_index(drop=True),
    tokenizer, MAX_LEN,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=(device.type == 'cuda'))
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          num_workers=2, pin_memory=(device.type == 'cuda'))

print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')


Train batches: 3568 | Test batches: 892


## 5. Fine-Tuning Loop

3 epochs with AdamW (lr = 2e-5) — standard BERT fine-tuning recipe.

In [ ]:
EPOCHS = 3
optimizer = AdamW(model.parameters(), lr=2e-5)

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch_idx, batch in enumerate(train_loader):
        optimizer.zero_grad()

        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if (batch_idx + 1) % 200 == 0:
            print(f'  Epoch {epoch+1} | Batch {batch_idx+1}/{len(train_loader)} '
                  f'| Running loss: {total_loss/(batch_idx+1):.4f}')

    avg_loss = total_loss / len(train_loader)
    print(f'Epoch {epoch+1}/{EPOCHS} complete | Avg loss: {avg_loss:.4f}\n')


Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/opt/homebrew/Cellar/python@3.11/3.11.15/Frameworks/Python.framework/Versions/3.11/lib/python3.11/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Cellar/python@3.11/3.11.15/Frameworks/Python.framework/Versions/3.11/lib/python3.11/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: Can't get attribute 'FakeNewsDataset' on <module '__main__' (built-in)>


## 6. Evaluation

In [ ]:
model.eval()
preds, true_vals = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        true_vals.extend(batch['labels'].numpy())

print(f'Accuracy: {accuracy_score(true_vals, preds):.4f}\n')
print(classification_report(true_vals, preds, target_names=['Real (0)', 'Fake (1)']))


## 7. Save Model

In [ ]:
SAVE_PATH = '../bert-finetuned'
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f'Model and tokenizer saved to {SAVE_PATH}/')


## 8. Inference Helper

Load the saved model and classify a single piece of text.

In [ ]:
from transformers import pipeline

classifier = pipeline(
    'text-classification',
    model=SAVE_PATH,
    tokenizer=SAVE_PATH,
    device=0 if device.type == 'cuda' else -1,
)

samples = [
    'Scientists confirm new vaccine shows 95% efficacy in large-scale trial.',
    'BREAKING: government secretly puts microchips in drinking water!!',
]
for text in samples:
    result = classifier(text, truncation=True, max_length=128)[0]
    label  = 'FAKE' if result['label'] == 'LABEL_1' else 'REAL'
    print(f'{label} ({result["score"]:.3f})  |  {text[:80]}')
